In [ ]:
!pip install smolagents python-dotenv sqlalchemy --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.4/148.4 kB 3.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

my_token = userdata.get('HF_TOKEN')
with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={my_token}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [ ]:
from sqlalchemy import create_engine, inspect, text

db_path = "sqlite:///shakespeare.sqlite"
db_name = "shakespeare.sqlite"

if not os.path.exists("shakespeare.sqlite"):
    print("WARNING: db not found.")

engine = create_engine(db_path)

In [ ]:
inspector = inspect(engine)
table_names = inspector.get_table_names()

schema = "Database Schema:\n"

for table in table_names:
    schema += f"Table: {table}\n"
    columns = inspector.get_columns(table)
    for col in columns:
        schema += f"  - {col['name']} ({col['type']})\n"

print(schema)

Database Schema:
Table: chapters
  - id (INTEGER)
  - Act (INTEGER)
  - Scene (INTEGER)
  - Description (TEXT)
  - work_id (INTEGER)
Table: characters
  - id (INTEGER)
  - CharName (TEXT)
  - Abbrev (TEXT)
  - Description (TEXT)
Table: paragraphs
  - id (INTEGER)
  - ParagraphNum (INTEGER)
  - PlainText (TEXT)
  - character_id (INTEGER)
  - chapter_id (INTEGER)
Table: works
  - id (INTEGER)
  - Title (TEXT)
  - LongTitle (TEXT)
  - Date (INTEGER)
  - GenreType (TEXT)



In [ ]:
from smolagents import tool
@tool
def sql_engine(query: str) -> str:
    """
    Allows you to perform SQL queries on the table. Returns a string representation of the result.

    Args:
        query: The query to perform.
    """
    output = ""
    with engine.connect() as con:
        rows = con.execute(text(query))
        for row in rows:
            output += "\n" + str(row)
    return output

In [ ]:
from smolagents import CodeAgent, InferenceClientModel

agent = CodeAgent(
    tools=[sql_engine],
    model=InferenceClientModel(model_id="Qwen/Qwen3-8B"),
)

In [ ]:
question = "Give the title and the characters name of the most recent work of Shakespeare."
evidence = "characters name refers to CharName; most recent work refers to max(Date)"
instructions = """INSTRUCTIONS:
- You can use the 'sql_engine' tool to make exploratory queries on the database.
- Do not execute separate queries to fetch values and then inject them into a subsequent query. If you need to filter by a dynamic value, you have to use a SQL subquery.
- Your SQL query must yield the answer on its own. You can not manipulate the data with Python.
"""

In [ ]:
USER_PROMPT = f"""
Database Schema:
{schema}
Question:
{evidence}. {question}
{instructions}
"""

#For each query, you can use the 'sql_engine' tool to analyze the output of the query.
#The final answer, that you return using the 'final_answer' tool, has to contain the final SQL query, not the answer to the question.

#Eventually, provide ONLY the final SQL query enclosed within the tags <answer> and </answer>, not the answer to the query.
#Once the final, correct SQL query is constructed, execute the final step by calling final_answer(your_query_string) within the <code> block.




In [ ]:
agent.run(USER_PROMPT)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Database Schema:                                                                                                │
│ Database Schema:                                                                                                │
│ Table: chapters                                                                                                 │
│   - id (INTEGER)                                                                                                │
│   - Act (INTEGER)                                                                                               │
│   - Scene (INTEGER)                                                                                             │
│   - Description (TEXT)                                                                                          │
│   - work_id (INTEGER)                                                                                           │
│ Table: characters                                                                                               │
│   - id (INTEGER)                                                                                                │
│   - CharName (TEXT)                                                                                             │
│   - Abbrev (TEXT)                                                                                               │
│   - Description (TEXT)                                                                                          │
│ Table: paragraphs                                                                                               │
│   - id (INTEGER)                                                                                                │
│   - ParagraphNum (INTEGER)                                                                                      │
│   - PlainText (TEXT)                                                                                            │
│   - character_id (INTEGER)                                                                                      │
│   - chapter_id (INTEGER)                                                                                        │
│ Table: works                                                                                                    │
│   - id (INTEGER)                                                                                                │
│   - Title (TEXT)                                                                                                │
│   - LongTitle (TEXT)                                                                                            │
│   - Date (INTEGER)                                                                                              │
│   - GenreType (TEXT)                                                                                            │
│                                                                                                                 │
│ Question:                                                                                                       │
│ characters name refers to CharName; most recent work refers to max(Date). Give the title and the characters     │
│ name of the most recent work of Shakespeare.                                                                    │
│ INSTRUCTIONS:                                                                                                   │
│ - You can use the 'sql_engine' tool to make exploratory queries on the database.                                │
│ - Do not execute separate queries to fetch values and then inject them into a subsequent query. If you need to  │
│ filter by a dynamic value, you have to use a SQL subqu

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = """                                                                                                      
  SELECT w.Title, c.CharName                                                                                       
  FROM works w                                                                                                     
  JOIN chapters ch ON w.id = ch.work_id                                                                            
  JOIN paragraphs p ON ch.id = p.chapter_id                                                                        
  JOIN characters c ON p.character_id = c.id                                                                       
  WHERE w.LongTitle LIKE '%Shakespeare%'                                                                           
  ORDER BY w.Date DESC                                                                                             
  LIMIT 1;                                                                                                         
  """                                                                                                              
  result = sql_engine(query=query)                                                                                 
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 

[Step 1: Duration 31.86 seconds| Input tokens: 2,324 | Output tokens: 2,080]

''

In [ ]:
import re

def get_clean_log_string(agent):

    log_parts = []
    final_query = None

    for i, step in enumerate(agent.memory.steps):

        if i == 0: continue # Skip the task step

        thought = getattr(step, 'model_output', getattr(step, 'thought', None))
        if thought:
            clean_thought = re.sub(r'<code>.*?</code>', '', thought, flags=re.DOTALL)
            clean_thought = clean_thought.replace('\n', ' ').strip()
            if clean_thought:
                log_parts.append(f"{clean_thought}")

        # call
        if hasattr(step, 'tool_calls') and step.tool_calls:
            for tool_call in step.tool_calls:
                content = getattr(tool_call, 'content', str(tool_call))

                # Regex to find SQL
                sql_match = re.search(r'query\s*=\s*"""(.*?)"""', content, re.DOTALL)
                if not sql_match:
                    sql_match = re.search(r"query\s*=\s*'''(.*?)'''", content, re.DOTALL)
                if not sql_match:
                    sql_match = re.search(r'query\s*=\s*"(.*?)"', content, re.DOTALL)

                if sql_match:
                    raw_sql = sql_match.group(1).strip()
                    # Flatten the SQL: remove newlines and double backslashes
                    flat_sql = raw_sql.replace('\n', ' ').replace('\\n', ' ').replace("   ", " ")

                    log_parts.append(f"[CALL] {flat_sql}")
                    final_query = flat_sql

        # ans - obs
        if hasattr(step, 'observations') and step.observations:
            obs = str(step.observations).strip()

            obs = obs.replace("Execution logs:", "").replace("Last output from code snippet:", "")
            obs = re.sub(r'\bNone\b', '', obs)
            obs = re.sub(r"^\('(.+)',\)$", r"\1", obs.strip(), flags=re.MULTILINE)
            obs = re.sub(r"^\('(.+)'\)$", r"\1", obs.strip(), flags=re.MULTILINE)

            obs_clean = obs.strip().replace('\n', ' ')

            # Truncate if too long
            if len(obs_clean) > 200:
                obs_clean = obs_clean[:200] + "... [truncated]"

            if obs_clean:
                log_parts.append(f"[ANS] {obs_clean}")

        # errors
        if hasattr(step, 'error') and step.error:
             err_clean = str(step.error).replace('\n', ' ')
             log_parts.append(f"[ERROR] {err_clean}")

    # Join everything with a single newline between steps
    full_log_string = " ".join(log_parts)

    return full_log_string, final_query


In [ ]:
log_string, pred_query = get_clean_log_string(agent)

pred_query = pred_query.replace("\\n", " ").replace("\\'", "'").strip()

print("LOG:\n" + log_string)
print(f"\nFinal Query: {pred_query}")

LOG:
Thought: To find the most recent work of Shakespeare, I need to identify the work with the maximum Date where the LongTitle includes "Shakespeare". Then, retrieve the title of that work and the associated character names. I'll use SQL joins to link the works to characters via chapters and paragraphs. [CALL]  SELECT w.Title, c.CharName FROM works w JOIN chapters ch ON w.id = ch.work_id JOIN paragraphs p ON ch.id = p.chapter_id JOIN characters c ON p.character_id = c.id WHERE w.LongTitle LIKE \'%Shakespeare%\' ORDER BY w.Date DESC LIMIT 1; 

Final Query: SELECT w.Title, c.CharName FROM works w JOIN chapters ch ON w.id = ch.work_id JOIN paragraphs p ON ch.id = p.chapter_id JOIN characters c ON p.character_id = c.id WHERE w.LongTitle LIKE '%Shakespeare%' ORDER BY w.Date DESC LIMIT 1;


In [ ]:
print(agent.memory.steps[1].model_input_messages[0].content[0]['text'])

You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.

Here are a few examples 

In [ ]:
def compute_execution_accuracy(gt_results, predict_results):
  num_correct = 0
  num_queries = len(gt_results)
  mismatch_idx = []

  for i, result in enumerate(gt_results):
      if set(result['results']) == set(predict_results[i]['results']):
          num_correct += 1
      else:
          mismatch_idx.append(i)

  acc = (num_correct / num_queries)

  return acc

In [ ]:
import sqlite3
def run_query(db_path, query):
  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()
  cursor.execute(query)
  rows = cursor.fetchall()
  conn.close()

  # Flatten results and convert to list of strings
  return [row[0] for row in rows]

In [ ]:
gt_query = """SELECT DISTINCT w.Title, ch.CharName FROM works w JOIN chapters c ON w.id = c.work_id JOIN paragraphs p ON p.chapter_id = c.id JOIN characters ch ON ch.id = p.character_id WHERE w.date = ( SELECT max(date) FROM works w2 )"""

In [ ]:
rows_gt = run_query(db_name, gt_query)
gt_res = [{"results": rows_gt}]

rows_pred = run_query(db_name, pred_query)
pred_res = [{"results": rows_pred}]

In [ ]:
acc = compute_execution_accuracy(gt_res, pred_res)
print(f"Accuracy of the generated SQL query: {acc}")

Accuracy of the generated SQL query: 0.0


In [ ]:
complete_trace = {
    "input": f"Question: {question}. Evidence: {evidence}. {instructions}",
    "output": log_string,
    "pred_query": pred_query,
    "target_query": gt_query,
    "execution_accuracy": int(acc)
}
complete_trace

{'input': "Question: Give the title and the characters name of the most recent work of Shakespeare.. Evidence: characters name refers to CharName; most recent work refers to max(Date). INSTRUCTIONS:\n- You can use the 'sql_engine' tool to make exploratory queries on the database.\n- Do not execute separate queries to fetch values and then inject them into a subsequent query. If you need to filter by a dynamic value, you have to use a SQL subquery.\n- Your SQL query must yield the answer on its own. You can not manipulate the data with Python.\n",
 'output': 'Thought: To find the most recent work of Shakespeare, I need to identify the work with the maximum Date where the LongTitle includes "Shakespeare". Then, retrieve the title of that work and the associated character names. I\'ll use SQL joins to link the works to characters via chapters and paragraphs. [CALL]  SELECT w.Title, c.CharName FROM works w JOIN chapters ch ON w.id = ch.work_id JOIN paragraphs p ON ch.id = p.chapter_id JOIN